<a href="https://colab.research.google.com/github/sgevatschnaider/data-science-business-decisions/blob/main/notebooks/10-clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background:linear-gradient(120deg,#07172c,#12395b 62%,#0d615d);padding:32px 35px;border-radius:22px;color:#f2fbff;border:1px solid #2bbdab;box-shadow:0 14px 40px #08233835">
<p style="font-size:13px;letter-spacing:3px;color:#73eedb;margin:0">UNIDAD III · MÓDULO 10 · LABORATORIO INTEGRAL</p>
<h1 style="font-size:36px;line-height:1.17;color:white;margin:16px 0">PCA, K-Means y<br>clustering jerárquico</h1>
<p style="font-size:17px;color:#c3e2ee;line-height:1.6">De la geometría de los datos a una segmentación verificable y accionable. Experimentos reproducibles, matemáticas, visualizaciones y decisiones de negocio.</p>
<div style="display:flex;gap:8px;flex-wrap:wrap;margin-top:20px"><span style="background:#1b5667;border-radius:20px;padding:7px 12px">PCA · 29 diapositivas</span><span style="background:#1b5667;border-radius:20px;padding:7px 12px">K-Means · 31 diapositivas</span><span style="background:#1b5667;border-radius:20px;padding:7px 12px">Jerárquico · 31 diapositivas</span><span style="background:#0b7267;border-radius:20px;padding:7px 12px">14 experimentos coordinados</span></div>
<p style="font-size:12px;color:#b3d0dd;margin-top:20px">Material elaborado por el profesor <b>Sergio Gevatschnaider</b> · Ciencia de Datos para Decisiones de Negocio · Colab / Jupyter · Python 3</p>
</div>

**Cómo usarlo:** en Colab elegí **Entorno de ejecución → Ejecutar todas**. El laboratorio produce datos sintéticos, figuras y reportes sin pedir archivos ni claves. Las secciones opcionales de interacción se adaptan si `ipywidgets` no está disponible.

[Guía y presentaciones](https://sgevatschnaider.github.io/data-science-business-decisions/modulos/10-clustering/guia.html) · [14 simulaciones HTML](https://sgevatschnaider.github.io/data-science-business-decisions/modulos/10-clustering/simulacion.html) · [Glosario (132 conceptos)](https://sgevatschnaider.github.io/data-science-business-decisions/modulos/10-clustering/glosario.html) · [Cuestionario (128 preguntas)](https://sgevatschnaider.github.io/data-science-business-decisions/modulos/10-clustering/cuestionario.html)

## Itinerario de estudio y objetivos

| Bloque | Qué vas a construir y comprobar | Diapositivas / simulaciones |
|:--|:--|:--|
| **00 · Preparación** | Entorno, semilla, datos RFM sintéticos, reglas de distancia y errores de escala | Transversal |
| **01 · PCA** | SVD y autovectores, loadings, scores, varianza, biplot, reconstrucción y atípicos | Clase 1 · S01–S04 |
| **02 · K-Means** | Lloyd manual, centroides, inercia, K-Means++, semillas, elbow, silhouette y estabilidad | Clase 2 · S05–S09 |
| **03 · Jerárquico** | Distancias, linkage simple/completo/promedio/Ward, dendrogramas, cortes y perfiles | Clase 3 · S10–S14 |
| **04 · Contrastes** | DBSCAN, lunas, ruido, sensibilidad y limitaciones de validación | Clases 2 y 3 |
| **05 · Caso de negocio** | Segmentación RFM de clientes, perfiles, acciones, asignación de nuevos clientes y monitoreo | Integración de 91 diapositivas |
| **06 · Evaluación** | Comprobaciones automáticas, preguntas de desarrollo, entregables y registro de decisión | Transferencia profesional |

**Resultado esperado:** defender por qué se eligieron variables, transformaciones, algoritmo, número de grupos y una acción concreta; separar **evidencia geométrica** de **valor empresarial**, y explicitar los riesgos.

**Criterio de éxito:** reproducibilidad con semilla fija, calidad comparada con referencias, ausencia de fugas, estabilidad razonable, perfiles interpretables y acciones comprobables. El mejor valor de silhouette **no constituye por sí solo** una decisión empresarial correcta.

<div style="background:#0b2439;color:#edf7ff;padding:16px 21px;border-left:5px solid #48dfc5;border-radius:12px">
<b>Guía de ejecución</b><br>1) Ejecutá en orden, sin saltar celdas. 2) Modificá los parámetros marcados <code>EXPERIMENTO</code>. 3) Explicá qué cambia y qué se mantiene. 4) Guardá tus decisiones al final.
<br><small style="color:#9fcedf">Los colores en las figuras identifican clases calculadas, no etiquetas reales. Permutar los nombres de los grupos no altera la partición.</small>
</div>

## 00 · Ambiente y visualizaciones reproducibles

La reproducibilidad exige registrar versiones, semillas y decisiones de preprocesamiento. Este material evita descargas de datasets para ejecutarse también sin conexión de red (una vez que están disponibles las bibliotecas estándar de Colab). Las dependencias son `numpy`, `pandas`, `matplotlib`, `scikit-learn` y `scipy`.

In [ ]:
import json, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Ellipse
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist, squareform, cdist
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (silhouette_score, silhouette_samples,
     adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score)
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import NearestNeighbors
import sklearn, scipy

SEED=42
rng=np.random.default_rng(SEED)
COL={'navy':'#10263d','blue':'#5286e4','mint':'#0f938d','amber':'#e7a13a',
     'pink':'#dd699b','muted':'#74879b','light':'#f5f8fc'}
COLORS=['#187b85','#e4a13b','#637ad3','#bd6ca2','#76a549','#df756b','#8292aa']
plt.rcParams.update({'figure.figsize':(10,4.5),'figure.dpi':108,'font.size':10.3,
 'axes.titlesize':12,'axes.titleweight':'bold','axes.labelcolor':COL['navy'],
 'axes.spines.top':False,'axes.spines.right':False,'grid.alpha':.19,
 'savefig.bbox':'tight','figure.facecolor':'white','axes.facecolor':'white'})

def scatter_groups(ax,X,labels,title='',centers=None,s=19,alpha=.85):
    for i,g in enumerate(np.unique(labels)):
        mask=np.asarray(labels)==g
        ax.scatter(X[mask,0],X[mask,1],s=s,alpha=alpha,
                   c=('#a0a5b2' if g==-1 else COLORS[int(g)%len(COLORS)]),
                   label=('ruido' if g==-1 else f'grupo {int(g)+1}'),edgecolors='none')
    if centers is not None:
        ax.scatter(centers[:,0],centers[:,1],marker='X',s=190,
                   c='white',edgecolors=COL['navy'],linewidths=1.7,zorder=5)
    ax.set_title(title);ax.set_xlabel('Eje 1');ax.set_ylabel('Eje 2');ax.grid(True)

def audit_table(df,digits=3):
    from IPython.display import display
    display(df.round(digits))

versions={'Python':platform.python_version(),'numpy':np.__version__,
 'pandas':pd.__version__,'scikit-learn':sklearn.__version__,
 'scipy':scipy.__version__,'matplotlib':matplotlib.__version__,'semilla':SEED}
print('ENTORNO VERIFICADO');print(json.dumps(versions,ensure_ascii=False,indent=2))

### Experimento 00 · ¿Cómo una variable altera una distancia?

Si no hay etiquetas, **la métrica es parte de la hipótesis**: distancia euclídea, Manhattan, coseno o una distancia de Gower para variables mixtas no son equivalentes. La escala puede eclipsar variables sustantivas. Una distancia de 1 en pesos no puede compararse directamente con 1 en frecuencia sin justificar la ponderación.

$$d_2(x_i,x_j)=\sqrt{\sum_{r=1}^{p}(x_{ir}-x_{jr})^2}$$

La estandarización $z=(x-\mu)/s$ modifica esa geometría. Es una transformación, **no una garantía** de grupos útiles.

In [ ]:
mini=pd.DataFrame({'visitas_mes':[2,3,16,18], 'gasto_mensual':[1800,2200,1700,2150]},
                  index=['A','B','C','D'])
from scipy.spatial.distance import pdist, squareform
raw_dist=squareform(pdist(mini,metric='euclidean'))
zmini=StandardScaler().fit_transform(mini)
scaled_dist=squareform(pdist(zmini))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,data,title in zip(axes,[raw_dist,scaled_dist],['Distancia cruda: domina el gasto','Distancia estandarizada: cambia la proximidad']):
    im=ax.imshow(data,cmap='YlGnBu');ax.set_title(title)
    ax.set_xticks(range(4),mini.index);ax.set_yticks(range(4),mini.index)
    for (i,j),v in np.ndenumerate(data):ax.text(j,i,f'{v:.1f}',ha='center',va='center',fontsize=9)
    fig.colorbar(im,ax=ax,shrink=.75)
fig.tight_layout();plt.show()
assert not np.allclose(raw_dist/raw_dist.max(),scaled_dist/scaled_dist.max())
audit_table(mini)

**Interpretación:** A y B tienen gastos próximos, mientras C y D tienen visitas próximas; en escala original el gasto tiende a determinar la distancia. **Pregunta:** ¿conviene una variable de valor monetario ponderada más que frecuencia? La respuesta es una decisión de negocio, no del algoritmo.

---
## 01 · PCA: de la covarianza a la reconstrucción

PCA obtiene direcciones ortogonales que maximizan varianza sucesivamente. Para datos centrados $X_c$, la descomposición $X_c=U\Sigma V^T$ determina las direcciones de los componentes y su varianza: $\lambda_j=\sigma_j^2/(n-1)$. Los *scores* son coordenadas proyectadas; los *loadings* describen qué variables contribuyen a cada eje. **No son etiquetas de clusters**.

### Experimento 01 · Una nube rotada y el eje de máxima varianza (S01)

Al rotar la dirección de proyección, la varianza proyectada $\mathrm{Var}(X_c v)$ tiene máximo sobre el primer autovector de la covarianza, sujeto a $\lVert v\rVert=1$. El signo de cada componente es arbitrario: $v$ y $-v$ expresan el mismo eje.

In [ ]:
rr=np.random.default_rng(123)
a=rr.normal(size=270);X2=np.column_stack([2.8*a+.38*rr.normal(size=270),1.65*a+.52*rr.normal(size=270)])
X2=X2-X2.mean(axis=0)
angles=np.linspace(0,np.pi,181)
var_angles=np.array([np.var(X2@np.array([np.cos(t),np.sin(t)]),ddof=1) for t in angles])
p2=PCA(n_components=2).fit(X2)
opt=np.degrees(np.arctan2(p2.components_[0,1],p2.components_[0,0]))%180
fig,axes=plt.subplots(1,2,figsize=(12,4.3))
axes[0].scatter(X2[:,0],X2[:,1],s=13,c=COL['blue'],alpha=.45)
for j,vec in enumerate(p2.components_):
    axes[0].arrow(0,0,* (vec*np.sqrt(p2.explained_variance_[j])*2.1),
             color=COLORS[j],width=.035,head_width=.22,length_includes_head=True)
axes[0].set(aspect='equal',title='PC1 y PC2 sobre los datos',xlabel='Variable 1 centrada',ylabel='Variable 2 centrada');axes[0].grid(True)
axes[1].plot(np.degrees(angles),var_angles,color=COL['mint'],lw=2.4)
axes[1].axvline(opt,color=COL['pink'],ls='--',label=f'PC1 ~ {opt:.1f}°')
axes[1].set(title='Varianza según ángulo de proyección',xlabel='Ángulo (grados)',ylabel='Varianza proyectada');axes[1].legend();axes[1].grid(True)
fig.tight_layout();plt.show()
print(f'Ángulo PC1: {opt:.1f}°; varianza PC1={p2.explained_variance_[0]:.3f}; PC2={p2.explained_variance_[1]:.3f}')
assert abs(max(var_angles)-p2.explained_variance_[0])<.003

### Experimento 02 · Centrar, escalar o no escalar (S02)

PCA sobre la **covarianza** privilegia variables de mayor dispersión; PCA sobre datos estandarizados equivale a PCA sobre la **matriz de correlaciones** (con la convención de normalización correspondiente). La decisión depende de unidades e interpretación; estandarizar ciegamente puede darle peso excesivo al ruido.

In [ ]:
ra=np.random.default_rng(7)
base=ra.normal(size=350)
scale_data=pd.DataFrame({'gasto_ARS':50000+18000*base+ra.normal(0,2500,350),
 'frecuencia':4+.9*base+ra.normal(0,.3,350),'reclamos':ra.poisson(2,350)})
rawp=PCA(n_components=3).fit(scale_data)
scaler_demo=StandardScaler().fit(scale_data)
stdp=PCA(n_components=3).fit(scaler_demo.transform(scale_data))
res=pd.DataFrame({'PCA sin escalar':rawp.explained_variance_ratio_,
                  'PCA estandarizado':stdp.explained_variance_ratio_},index=['PC1','PC2','PC3'])
audit_table(res)
fig,ax=plt.subplots(figsize=(10,3.5))
res.plot(kind='bar',ax=ax,color=[COL['amber'],COL['mint']],rot=0)
ax.set(title='La escala altera cuánta varianza explica cada componente',ylabel='Proporción de varianza');ax.grid(axis='y');plt.show()
assert rawp.explained_variance_ratio_[0]>.99

### Experimento 03 · PCA por SVD y contraste con autovectores (S01–S03)

Para $X$ centrado, $S=X^TX/(n-1)$ es una matriz simétrica semidefinida positiva. Sus autovectores forman la base principal; por SVD se evita construir explícitamente $S$, lo que puede mejorar la estabilidad numérica. Verificamos igualdad de autovalores y reconstrucción, permitiendo ambigüedad de signo.

In [ ]:
Xbase=StandardScaler().fit_transform(scale_data)
U,singular,Vt=np.linalg.svd(Xbase-Xbase.mean(axis=0),full_matrices=False)
lam_svd=singular**2/(len(Xbase)-1)
lam_eig,eigvec=np.linalg.eigh(np.cov(Xbase,rowvar=False))
lam_eig=lam_eig[::-1]
pca_svd=PCA(svd_solver='full').fit(Xbase)
check=pd.DataFrame({'λ desde SVD':lam_svd,'λ desde covarianza':lam_eig,
                    'λ sklearn':pca_svd.explained_variance_})
audit_table(check)
assert np.allclose(lam_svd,lam_eig,rtol=1e-9)
assert np.allclose(lam_eig,pca_svd.explained_variance_,rtol=1e-9)
print('Verificación SVD, covarianza y sklearn: OK')

### Experimento 04 · Scree plot y error de reconstrucción (S03)

La varianza explicada acumulada crece con $q$, pero retener el 95 % no implica conservar variables discriminantes, grupos raros ni relaciones no lineales. Para una reconstrucción $\hat X_q=T_q V_q$, el error cuadrático decrece al añadir componentes. Elegí $q$ por compromiso entre compresión, pérdida, interpretabilidad y desempeño de la tarea posterior.

In [ ]:
from sklearn.datasets import make_classification
Xhi,_=make_classification(n_samples=440,n_features=9,n_informative=4,n_redundant=3,n_repeated=0,
                           n_clusters_per_class=1,class_sep=1.15,random_state=SEED)
Xhi=StandardScaler().fit_transform(Xhi)
pca9=PCA().fit(Xhi)
cum=np.cumsum(pca9.explained_variance_ratio_)
mses=[]
for q in range(1,Xhi.shape[1]+1):
    model=PCA(n_components=q).fit(Xhi)
    mses.append(np.mean((Xhi-model.inverse_transform(model.transform(Xhi)))**2))
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(np.arange(1,10),pca9.explained_variance_ratio_,color=COL['blue'],alpha=.65)
axes[0].plot(np.arange(1,10),cum,'o-',color=COL['mint'],label='acumulada')
axes[0].axhline(.9,color=COL['pink'],ls='--',label='referencia 90%')
axes[0].set(title='Scree plot y varianza acumulada',xlabel='Componentes',ylabel='Fracción');axes[0].legend();axes[0].grid(axis='y')
axes[1].plot(range(1,10),mses,'o-',color=COL['amber'])
axes[1].set(title='Reconstrucción al conservar q componentes',xlabel='q',ylabel='Error cuadrático medio');axes[1].grid(True)
fig.tight_layout();plt.show()
assert all(np.diff(mses)<1e-8)
print('Componentes necesarios para >= 90% de varianza:',int(np.searchsorted(cum,.9)+1))

### Experimento 05 · Biplot y loadings (S04)

Un *biplot* combina scores (filas) y flechas relacionadas con la estructura de las variables. El signo relativo de las flechas, su ángulo y longitud merecen una lectura prudente: **la escala del dibujo no reemplaza el cálculo**. Importante: PCA no es una técnica de agrupamiento.

In [ ]:
feature_names=['gasto','visitas','antiguedad','devoluciones','uso_digital']
z=np.random.default_rng(91)
latent=z.normal(size=(350,2))
X5=np.column_stack([2.0*latent[:,0]+.2*z.normal(size=350),
  1.7*latent[:,0]+.3*z.normal(size=350),
  -1.2*latent[:,1]+.4*z.normal(size=350),
  -1.0*latent[:,0]+.7*z.normal(size=350),
  1.8*latent[:,1]+.4*z.normal(size=350)])
X5=StandardScaler().fit_transform(X5)
p5=PCA(n_components=2).fit(X5);scores=p5.transform(X5)
loadings=p5.components_.T * np.sqrt(p5.explained_variance_)
fig,ax=plt.subplots(figsize=(8.5,6))
ax.scatter(scores[:,0],scores[:,1],s=11,c=COL['blue'],alpha=.24,label='observaciones')
for i,feat in enumerate(feature_names):
    dx,dy=loadings[i]*2.4
    ax.annotate('',xy=(dx,dy),xytext=(0,0),arrowprops=dict(arrowstyle='-|>',color=COLORS[i],lw=2))
    ax.text(dx*1.15,dy*1.15,feat,color=COLORS[i],ha='center',fontsize=10,weight='bold')
ax.set(title='Biplot didáctico: scores + direcciones de variables',xlabel=f'PC1 ({p5.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({p5.explained_variance_ratio_[1]:.1%})');ax.grid(True);plt.show()
audit_table(pd.DataFrame(p5.components_.T,index=feature_names,columns=['PC1 loading (coef.)','PC2 loading (coef.)']))

### Experimento 06 · Puntos atípicos, influencia y PCA robusta (S04)

PCA ordinaria utiliza cuadrados implícitos en la covarianza; unos pocos valores extremos pueden rotar su primer eje. `RobustScaler` corrige la localización y escala por mediana y rango intercuartílico, pero **no convierte PCA clásica en PCA robusta**. Se requiere contrastar una muestra depurada o técnicas especiales; nunca eliminar casos válidos solo para embellecer un gráfico.

In [ ]:
base2=np.random.default_rng(23).multivariate_normal([0,0],[[3.5,2.8],[2.8,2.9]],size=230)
out=np.array([[-1,13],[0,14],[1,13.5],[.3,12]])
with_out=np.vstack([base2,out])
fit_clean=PCA(n_components=2).fit(base2)
fit_out=PCA(n_components=2).fit(with_out)
def comp_angle(v):return np.degrees(np.arctan2(v[1],v[0]))%180
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,X,p,title in zip(axes,[base2,with_out],[fit_clean,fit_out],['Sin valores atípicos','Con 4 observaciones extremas']):
    ax.scatter(X[:,0],X[:,1],s=13,alpha=.4,c=COL['blue'])
    vec=p.components_[0]*4
    ax.plot([-vec[0],vec[0]],[-vec[1],vec[1]],color=COL['pink'],lw=3)
    ax.set(xlim=(-7,7),ylim=(-8,16),title=f'{title}\nPC1: {comp_angle(p.components_[0]):.1f}°');ax.grid(True)
fig.tight_layout();plt.show()
assert abs(comp_angle(fit_clean.components_[0])-comp_angle(fit_out.components_[0]))>4

---
## 02 · K-Means: objetivo, iteraciones y validación

Para una partición $C_1,\dots,C_K$, K-Means minimiza la inercia (WCSS)

$$J=\sum_{j=1}^{K}\sum_{i\in C_j}\lVert x_i-\mu_j\rVert_2^2,\qquad \mu_j=\frac{1}{|C_j|}\sum_{i\in C_j}x_i.$$

El algoritmo de Lloyd alterna **asignación** y **actualización**. Con asignaciones consistentes y centros medios, $J$ no aumenta por la actualización de cada etapa completa; pero converge a un óptimo **local**. La inercia siempre tiende a disminuir al aumentar $K$: no justifica por sí misma una elección.

### Experimento 07 · Lloyd desde cero (S05–S06)

Veremos las fronteras inducidas por los centros, la secuencia de descenso de inercia y su condición de parada. El ejemplo no oculta la distancia ni usa `fit()` para mostrar el mecanismo.

In [ ]:
Xk,y_truth=make_blobs(n_samples=360,centers=[[-3,-2],[0,3],[3,-1],[4,4]],
                 cluster_std=[.65,.88,.7,.8],random_state=SEED)
Xk=StandardScaler().fit_transform(Xk)

def lloyd_steps(X,k=4,seed=11,max_iter=35):
    r=np.random.default_rng(seed)
    centers=X[r.choice(len(X),size=k,replace=False)].copy()
    hist=[]; last_labels=None
    for iteration in range(max_iter):
        dist=cdist(X,centers,'sqeuclidean')
        labels=np.argmin(dist,axis=1)
        before=float(np.sum(dist[np.arange(len(X)),labels]))
        new=centers.copy()
        for j in range(k):
            if np.any(labels==j):new[j]=X[labels==j].mean(axis=0)
        after=float(np.sum(np.min(cdist(X,new,'sqeuclidean'),axis=1)))
        hist.append({'iteracion':iteration+1,'inercia_antes':before,'inercia_despues':after,
                     'centros':new.copy(),'etiquetas':labels.copy()})
        if np.allclose(centers,new,atol=1e-9):break
        centers=new
    return centers,hist

centers,hist=lloyd_steps(Xk,4)
assert np.all(np.diff([h['inercia_despues'] for h in hist])<=1e-8)
model_k=KMeans(n_clusters=4,n_init=30,random_state=SEED).fit(Xk)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,h,j in zip(axes[:2],[hist[0],hist[-1]],[1,len(hist)]):
    scatter_groups(ax,Xk,h['etiquetas'],f'Iteración {j}',centers=h['centros'],s=13)
axes[2].plot(range(1,len(hist)+1),[h['inercia_despues'] for h in hist],'-o',color=COL['mint'])
axes[2].set(title='Inercia después de actualizar centroides',xlabel='Iteración',ylabel='WCSS');axes[2].grid(True)
fig.tight_layout();plt.show()
print(f'Lloyd manual: {len(hist)} iteraciones; WCSS={hist[-1]["inercia_despues"]:.2f}. sklearn: {model_k.inertia_:.2f}')

**Pregunta:** Si Lloyd converge, ¿eso demuestra que encontró el mínimo global? No: cambia la inicialización y repetí la experiencia. Cuidado con los clusters vacíos, el escalado y los outliers.

### Experimento 08 · Inicialización aleatoria, K-Means++ y reinicios (S07)

K-Means++ suele mejorar la dispersión inicial entre centroides, pero no asegura óptimo global. La comparación más justa mantiene **igual conjunto de datos, K, criterio, tolerancia y presupuesto de reinicios**.

In [ ]:
trial_rows=[]
for init in ('random','k-means++'):
    for seed in range(18):
        mk=KMeans(n_clusters=4,init=init,n_init=1,random_state=seed,max_iter=100).fit(Xk)
        trial_rows.append({'inicializacion':init,'semilla':seed,'inercia':mk.inertia_,'iteraciones':mk.n_iter_})
trials=pd.DataFrame(trial_rows)
audit_table(trials.groupby('inicializacion').agg(inercia_media=('inercia','mean'),
        inercia_min=('inercia','min'),inercia_max=('inercia','max'),iteraciones_medias=('iteraciones','mean')))
fig,ax=plt.subplots(figsize=(9,3.6))
for label,g in trials.groupby('inicializacion'):
    ax.scatter(g['semilla'],g['inercia'],label=label,s=39,alpha=.72)
ax.set(title='Variabilidad por inicialización: una sola corrida no es evidencia suficiente',xlabel='Semilla',ylabel='Inercia');ax.legend();ax.grid(True);plt.show()
assert trials['inercia'].notna().all()

### Experimento 09 · Elegir K: elbow, silhouette, Calinski–Harabasz y Davies–Bouldin (S08)

**Silhouette** compara la distancia intragrupo $a(i)$ con la menor distancia media hacia otro grupo $b(i)$:

$$s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}.$$

Se define para soluciones con al menos 2 grupos y menos grupos que observaciones. Debe interpretarse junto con tamaños, estabilidad y utilidad. Valores menores de Davies–Bouldin indican mejor separación relativa; mayores de Calinski–Harabasz, mayor relación entre dispersión inter/intra grupos (ninguno garantiza sentido empresarial).

In [ ]:
ks=range(2,9)
metrics=[]
for k in ks:
    mod=KMeans(n_clusters=k,random_state=SEED,n_init=15).fit(Xk)
    labels=mod.labels_
    metrics.append({'K':k,'inercia':mod.inertia_,'silhouette':silhouette_score(Xk,labels),
        'Calinski_Harabasz':calinski_harabasz_score(Xk,labels),
        'Davies_Bouldin':davies_bouldin_score(Xk,labels),
        'menor_cluster':min(np.bincount(labels))})
kmmetrics=pd.DataFrame(metrics)
audit_table(kmmetrics)
fig,axes=plt.subplots(2,2,figsize=(12,7))
for ax,col,title,color in zip(axes.ravel(),['inercia','silhouette','Calinski_Harabasz','Davies_Bouldin'],
        ['Inercia (menor = mejor ajuste)','Silhouette (mayor = mejor)','Calinski–Harabasz (mayor)','Davies–Bouldin (menor)'],
        [COL['blue'],COL['mint'],COL['amber'],COL['pink']]):
    ax.plot(kmmetrics['K'],kmmetrics[col],'-o',color=color)
    ax.set(xlabel='K',title=title);ax.grid(True)
fig.tight_layout();plt.show()
assert len(kmmetrics)==7

### Experimento 10 · Perfiles de silhouette: mirar más allá del promedio (S08)

Un promedio oculta grupos muy heterogéneos. El gráfico de silhouette por cluster muestra anchura, heterogeneidad, valores negativos (asignación posiblemente problemática) y tamaño. La validación debe detectar si un pequeño segmento es real o un artefacto.

In [ ]:
labels=model_k.labels_
sl=silhouette_samples(Xk,labels)
fig,ax=plt.subplots(figsize=(9,5))
ycur=10
for k in np.unique(labels):
    v=np.sort(sl[labels==k]);n=len(v)
    ax.fill_betweenx(np.arange(ycur,ycur+n),0,v,color=COLORS[k],alpha=.75)
    ax.text(-.18,ycur+n/2,f'Grupo {k+1}: {n}',va='center',fontsize=9)
    ycur+=n+12
ax.axvline(float(sl.mean()),ls='--',color=COL['navy'],label=f'Promedio {sl.mean():.2f}')
ax.set(title='Silhouette individual: K=4',xlabel='Coeficiente de silhouette',ylabel='Clientes ordenados por grupo',yticks=[]);ax.legend();ax.grid(axis='x');plt.show()
print(f'Puntos con silhouette negativo: {int((sl<0).sum())} de {len(sl)}')

### Experimento 11 · Escalas, duplicaciones y outliers (S09)

K-Means y Ward no son invariantes a la escala; tampoco lo son a introducir dos copias de la misma variable. Los valores extremos desplazan centroides y aumentan inercia. La distancia euclídea induce fronteras de Voronoi lineales, por lo que geometrías alargadas, anillos o lunas pueden ser mal representadas.

In [ ]:
r=np.random.default_rng(501)
Xchange=np.column_stack([r.normal(size=250),r.normal(size=250)*2+r.choice([-3,3],250)])
noisy=Xchange.copy();noisy[:,0]*=40
variants={'sin escalar':noisy,'StandardScaler':StandardScaler().fit_transform(noisy),
          'duplicar eje X':np.column_stack([noisy,noisy[:,[0]]])}
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,(name,dat) in zip(axes,variants.items()):
    labels=KMeans(n_clusters=2,random_state=SEED,n_init=15).fit_predict(dat)
    scatter_groups(ax,Xchange,labels,title=name,s=14)
fig.tight_layout();plt.show()

---
## 03 · Clustering jerárquico: dendrogramas, distancias y enlaces

El clustering aglomerativo empieza con grupos unitarios y fusiona pares hasta construir una jerarquía. Una altura de dendrograma indica **la disimilitud usada en esa fusión**, no necesariamente una distancia directa entre dos puntos; con Ward corresponde al aumento de dispersión intragrupo según su formulación.

* `single`: mínima distancia entre pares de observaciones; puede producir *chaining*.
* `complete`: máxima distancia entre pares; favorece grupos relativamente compactos.
* `average`: media de distancias entre pares.
* `ward`: fusiona buscando minimizar el incremento de suma de cuadrados; requiere geometría euclídea.

### Experimento 12 · Matriz de distancias y dendrograma explicado (S10–S11)

Las fusiones dependen tanto de la **métrica** como de la **regla de enlace**. Con el mismo conjunto de puntos, los resultados pueden cambiar radicalmente. El dendrograma no establece automáticamente un número correcto de clusters.

In [ ]:
small=np.array([[0,0],[.7,.3],[1.3,.1],[3.9,3.7],[4.4,4.1],[7.6,.2],[8.2,.4],[8.8,.1]],dtype=float)
names=list('ABCDEFGH')
D=squareform(pdist(small,'euclidean'))
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
for i,p in enumerate(small):axes[0].scatter(*p,c=COLORS[i%len(COLORS)],s=90);axes[0].text(p[0]+.1,p[1]+.12,names[i],weight='bold')
axes[0].set(title='Observaciones individuales',xlabel='Característica 1',ylabel='Característica 2');axes[0].grid(True)
im=axes[1].imshow(D,cmap='YlGnBu');axes[1].set(title='Matriz de distancias euclídeas',xticks=range(8),yticks=range(8));axes[1].set_xticklabels(names);axes[1].set_yticklabels(names)
for (i,j),v in np.ndenumerate(D):axes[1].text(j,i,f'{v:.1f}',ha='center',va='center',fontsize=8)
fig.colorbar(im,ax=axes[1],shrink=.7);fig.tight_layout();plt.show()
audit_table(pd.DataFrame(D,index=names,columns=names))

### Experimento 13 · Cuatro linkages y efecto chaining (S12–S14)

Comparamos funciones de SciPy sobre exactamente las mismas observaciones. En un dendrograma de Ward, alturas no deben interpretarse de forma idéntica a `single`. Probá modificar el corte horizontal y explicá qué pasa con el número de segmentos y su interpretación.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(13,8))
Zs={}
for ax,method in zip(axes.ravel(),['single','complete','average','ward']):
    Z=linkage(small,method=method,metric='euclidean')
    Zs[method]=Z
    dendrogram(Z,labels=names,ax=ax,color_threshold=None,
               above_threshold_color=COL['blue'],leaf_rotation=0)
    ax.set(title=f'Linkage: {method}',xlabel='Observación',ylabel='Altura de fusión');ax.grid(axis='y')
fig.tight_layout();plt.show()
assert all(z.shape==(len(small)-1,4) for z in Zs.values())

### Experimento 14 · Corte a K grupos y discrepancias entre métodos (S13)

Al cortar un dendrograma en $K$ grupos, evitamos asumir que dos métodos con mismo $K$ producen idéntica partición. Para medir discrepancias entre particiones —sin depender de números arbitrarios de etiquetas— usamos **ARI** (*Adjusted Rand Index*). Un ARI cercano a 1 refleja concordancia, cerca de 0 la esperada por azar bajo su modelo de referencia, y puede ser negativo.

In [ ]:
sets={method:fcluster(z,t=3,criterion='maxclust') for method,z in Zs.items()}
comparisons=[]
for a in sets:
    for b in sets:
        if a<b:comparisons.append({'Método A':a,'Método B':b,'ARI':adjusted_rand_score(sets[a],sets[b])})
audit_table(pd.DataFrame(comparisons))
fig,axes=plt.subplots(1,4,figsize=(16,3.7))
for ax,(method,labels) in zip(axes,sets.items()):
    scatter_groups(ax,small,labels-1,title=f'{method} · {len(np.unique(labels))} grupos',s=85)
fig.tight_layout();plt.show()

### Experimento 15 · Distancia cophenetic y lectura estructural

La distancia cophenetic de dos puntos es la altura a la que se unen por primera vez en el árbol. Su correlación con la distancia original describe qué tanto la jerarquía representa las disimilitudes, **no valida por sí misma segmentos útiles**. Un árbol con alta correlación puede fallar en estabilidad o accionabilidad.

In [ ]:
orig_d=pdist(Xk[:55])
quality=[]
for m in ['single','complete','average','ward']:
    Z=linkage(Xk[:55],method=m)
    corr,_=cophenet(Z,orig_d)
    labs=fcluster(Z,t=4,criterion='maxclust')
    quality.append({'método':m,'correlación_cophenetic':corr,
                    'silhouette_4':silhouette_score(Xk[:55],labs) if len(set(labs))>1 else np.nan})
audit_table(pd.DataFrame(quality))

---
## 04 · Cuando K-Means no sirve: densidad, ruido y DBSCAN

DBSCAN distingue puntos *core*, borde y ruido, formando grupos conectados por densidad. Sus parámetros `eps` y `min_samples` deben discutirse según escala, dimensión y densidad. Es valioso para formas no convexas, pero puede fallar si coexisten densidades muy diferentes.

### Experimento 16 · Lunas, outliers y comparación DBSCAN / K-Means

Dos lunas son un contraejemplo clásico: K-Means puede dividirlas geométricamente por rectas; DBSCAN puede capturar las formas curvas. La comparación evita usar etiquetas verdaderas para entrenar; solo se observan al analizar la geometría.

In [ ]:
Xm,truth=make_moons(n_samples=390,noise=.065,random_state=SEED)
rr=np.random.default_rng(32);Xm=np.vstack([Xm,rr.uniform(low=[-1.2,-.8],high=[2.2,1.5],size=(18,2))])
truth=np.concatenate([truth,np.full(18,-1)])
km=KMeans(n_clusters=2,n_init=20,random_state=SEED).fit_predict(Xm)
db=DBSCAN(eps=.17,min_samples=6).fit_predict(Xm)
fig,axes=plt.subplots(1,3,figsize=(14,4))
scatter_groups(axes[0],Xm,truth,'Estructura simulada (referencia)',s=12)
scatter_groups(axes[1],Xm,km,'K-Means: fronteras lineales',s=12)
scatter_groups(axes[2],Xm,db,'DBSCAN: grupos por densidad',s=12)
fig.tight_layout();plt.show()
print('DBSCAN:',len(set(db)-{-1}),'clusters; ruido detectado:',sum(db==-1),'observaciones')

### Experimento 17 · Sensibilidad de DBSCAN a ε y min_samples

Una buena elección no surge de maximizar el número de grupos. Un gráfico de distancia al vecino número `min_samples` ayuda a detectar transiciones, pero es una **heurística**, no una solución única. Con dimensiones altas, las distancias pueden perder contraste y requerir otro enfoque.

In [ ]:
min_pts=6
near=NearestNeighbors(n_neighbors=min_pts).fit(Xm)
kdist=np.sort(near.kneighbors(Xm)[0][:,-1])
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(kdist,color=COL['mint']);axes[0].set(title=f'Distancia al {min_pts}° vecino',xlabel='Observaciones ordenadas',ylabel='Distancia');axes[0].grid(True)
results=[]
for eps in [.10,.14,.17,.21,.27]:
    pred=DBSCAN(eps=eps,min_samples=min_pts).fit_predict(Xm)
    results.append({'eps':eps,'grupos':len(set(pred)-{-1}),'ruido':int(np.sum(pred==-1))})
eps_df=pd.DataFrame(results);audit_table(eps_df)
axes[1].plot(eps_df.eps,eps_df.grupos,'o-',color=COL['blue'],label='grupos')
axes[1].plot(eps_df.eps,eps_df.ruido,'s--',color=COL['pink'],label='ruido (cantidad)')
axes[1].set(title='ε cambia la estructura detectada',xlabel='ε');axes[1].grid(True);axes[1].legend()
fig.tight_layout();plt.show()

---
## 05 · Estabilidad, validación y transferencia

La validación sin etiquetas utiliza evidencia complementaria. Para K-Means, contrastamos semillas, muestras y asignaciones de observaciones comunes. Para jerárquico, distintos linkages y cortes. Evitamos suponer que una validación interna demuestra recuperabilidad causal o rentabilidad.

### Experimento 18 · Estabilidad ante remuestreo

Se toman dos submuestras independientes del **mismo conjunto original**. Entrenamos en cada una y comparamos particiones sobre las filas comunes, sin reetiquetar arbitrariamente centros. Reportamos mediana y dispersión del ARI. Un resultado estable con datos artificiales no autoriza trasladarlo sin validación a otros clientes.

In [ ]:
def bootstrap_stability(X,k=4,reps=14,frac=.8,seed=15):
    r=np.random.default_rng(seed);scores=[]
    for _ in range(reps):
        a=np.sort(r.choice(len(X),size=int(len(X)*frac),replace=False))
        b=np.sort(r.choice(len(X),size=int(len(X)*frac),replace=False))
        common,ia,ib=np.intersect1d(a,b,return_indices=True)
        pred_a=KMeans(n_clusters=k,random_state=int(r.integers(1e8)),n_init=8).fit_predict(X[a])
        pred_b=KMeans(n_clusters=k,random_state=int(r.integers(1e8)),n_init=8).fit_predict(X[b])
        scores.append(adjusted_rand_score(pred_a[ia],pred_b[ib]))
    return np.array(scores)
stab=bootstrap_stability(Xk)
fig,ax=plt.subplots(figsize=(9,3.8));ax.plot(range(1,len(stab)+1),stab,'o-',color=COL['mint'])
ax.axhline(np.median(stab),color=COL['pink'],ls='--',label=f'Mediana ARI: {np.median(stab):.2f}')
ax.set(title='Estabilidad de K-Means al variar la muestra',xlabel='Réplica',ylabel='ARI de filas comunes',ylim=(-.1,1.06));ax.grid(True);ax.legend();plt.show()
print('ARI [mínimo, mediana, máximo]:',np.round([stab.min(),np.median(stab),stab.max()],3))
assert np.all(np.isfinite(stab))

### Experimento 19 · PCA → K-Means: cuándo ayuda y cuándo destruye información

Una proyección PCA es lineal y optimiza varianza total, no separación de grupos. Reducir variables redundantes puede simplificar el problema; reducir demasiado puede borrar diferencias importantes. Comparamos K-Means en el espacio escalado, en PC1 y en dos PCs, sin inventar una superioridad general.

In [ ]:
Xrich=np.column_stack([Xk,2*Xk[:,0]+np.random.default_rng(22).normal(0,.2,len(Xk)),
                        Xk[:,1]*.7+np.random.default_rng(23).normal(0,.12,len(Xk))])
Xrich=StandardScaler().fit_transform(Xrich)
rows=[]
spaces={'Original (4D)':Xrich,'PCA 1D':PCA(n_components=1).fit_transform(Xrich),
        'PCA 2D':PCA(n_components=2).fit_transform(Xrich)}
for name,space in spaces.items():
    km=KMeans(n_clusters=4,random_state=SEED,n_init=15).fit(space)
    rows.append({'espacio':name,'dimensiones':space.shape[1],
                 'silhouette':silhouette_score(space,km.labels_),
                 'ARI_con_simulacion':adjusted_rand_score(y_truth,km.labels_)})
audit_table(pd.DataFrame(rows))
print('Importante: las etiquetas de la simulación solo se usan como diagnóstico post hoc; nunca en K-Means.')

---
## 06 · Caso empresarial integral: segmentación RFM y acciones

La empresa ficticia **Andes Market** desea personalizar campañas según comportamiento de compra. El conjunto sintético contiene importe, frecuencia, recencia, descuentos, antigüedad y contacto digital. No contiene datos personales reales. El caso permite evaluar si un segmento se puede describir, identificar en producción y tratar mediante una acción con costo explícito.

**Precaución:** RFM es una representación de conducta; no demuestra causalidad. La respuesta de campañas necesita experimentos con grupos de control y consentimiento.

### Experimento 20 · Generador de clientes con perfiles latentes (solo para comprobar)

Los perfiles latentes se utilizan **únicamente para crear datos plausibles**; se excluyen del entrenamiento. Ninguna solución algorítmica está obligada a recuperarlos ni se permite utilizarlos para asignación de clientes. También se introducen colas monetarias y heterogeneidad.

In [ ]:
def make_customers(n=660,seed=42):
    r=np.random.default_rng(seed)
    latent=r.choice(4,size=n,p=[.31,.28,.23,.18])
    recency_mean=np.array([80,27,12,140]);freq_mean=np.array([2.8,6,11,1.4])
    ticket_mean=np.array([7000,13000,19500,4900]);digital_mean=np.array([.33,.69,.89,.18])
    recency=np.maximum(0,r.gamma(2.4,recency_mean[latent]/2.4)).round().astype(int)
    freq=np.maximum(1,r.poisson(freq_mean[latent])).astype(int)
    # El importe anual es positivamente sesgado y correlacionado con frecuencia.
    ticket=ticket_mean[latent]*r.lognormal(mean=-.08,sigma=.36,size=n)
    spending=np.maximum(500,np.round(freq*ticket,0))
    discount=np.clip(r.beta(2.0+0.45*latent,6.2,size=n),0,.95)
    tenure=np.maximum(1,np.round(r.gamma(2.1,7,size=n)+latent*3)).astype(int)
    digital=np.clip(r.normal(digital_mean[latent],.15,size=n),0,1)
    df=pd.DataFrame({'cliente_id':[f'C{i:04d}' for i in range(1,n+1)],
       'recencia_dias':recency,'compras_anuales':freq,'gasto_anual_ARS':spending,
       'descuento_fraccion':discount,'antiguedad_meses':tenure,'uso_digital':digital})
    return df,latent
clientes,_latent_private=make_customers()
assert clientes['cliente_id'].is_unique and not clientes.isna().any().any()
print('Clientes sintéticos:',len(clientes),'· variables de comportamiento:',clientes.shape[1]-1)
audit_table(clientes.head(8),2)

### Experimento 21 · Exploración y transformaciones justificadas

La recencia y el gasto son asimétricos; $\log(1+x)$ reduce influencia de valores extremos sin truncarlos arbitrariamente. Para frecuencia y monetario mantenemos las unidades originales en reportes y usamos variables transformadas únicamente en la métrica. Las transformaciones deben calibrarse **en entrenamiento** y aplicarse sin refit a nuevos clientes.

In [ ]:
features=['recencia_dias','compras_anuales','gasto_anual_ARS','descuento_fraccion',
          'antiguedad_meses','uso_digital']
def customer_transform(df):
    Z=df[features].astype(float).copy()
    for col in ['recencia_dias','compras_anuales','gasto_anual_ARS','antiguedad_meses']:
        Z[col]=np.log1p(Z[col])
    return Z
transformed=customer_transform(clientes)
fig,axes=plt.subplots(2,3,figsize=(14,7.2))
for ax,column in zip(axes.ravel(),features):
    ax.hist(clientes[column],bins=24,color=COL['blue'],alpha=.75,edgecolor='white')
    ax.set_title(column.replace('_',' '));ax.grid(axis='y')
fig.suptitle('Exploración RFM: escalas y asimetrías ANTES de agrupar',fontweight='bold')
fig.tight_layout();plt.show()
print('Correlaciones entre variables transformadas:')
audit_table(transformed.corr())

### Experimento 22 · Entrenamiento, elección K y comparación jerárquica

Una segmentación es sensible a K, la selección de variables y el escalador. Este ejercicio hace visible cómo una decisión afecta tanto la métrica interna como los perfiles, sin afirmar que la métrica determine por sí sola el número óptimo. Para comparar jerárquico con Ward usamos el mismo espacio estandarizado.

In [ ]:
rfm_scaler=StandardScaler().fit(transformed)
Xrfm=rfm_scaler.transform(transformed)
rows=[]
for k in range(2,8):
    obj=KMeans(n_clusters=k,n_init=15,random_state=SEED).fit(Xrfm)
    rows.append({'K':k,'inercia':obj.inertia_,'silhouette':silhouette_score(Xrfm,obj.labels_),
                 'mínimo_grupo':int(np.bincount(obj.labels_).min())})
rfm_metrics=pd.DataFrame(rows)
audit_table(rfm_metrics)
K_BUSINESS=4  # EXPERIMENTO: cambia K entre 2 y 7 e interpreta perfiles
seg_model=KMeans(n_clusters=K_BUSINESS,n_init=35,random_state=SEED).fit(Xrfm)
clusters=seg_model.labels_
labels_ward=AgglomerativeClustering(n_clusters=K_BUSINESS,linkage='ward').fit_predict(Xrfm)
print('Silhouette K-Means:',round(silhouette_score(Xrfm,clusters),3),
      '| Ward:',round(silhouette_score(Xrfm,labels_ward),3),
      '| concordancia ARI:',round(adjusted_rand_score(clusters,labels_ward),3))
assert len(np.unique(clusters))==K_BUSINESS

### Experimento 23 · Perfiles, tamaño de grupos y acciones propuestas

Las medias en escala estandarizada no son unidades monetarias interpretables. Por eso informamos **medianas y tamaños originales**. Las etiquetas son identificadores matemáticos: para tomar decisiones deben convertirse en perfiles descriptivos sujetos a revisión. No hay causalidad entre pertenecer a un grupo y responder a una campaña.

In [ ]:
perfiles=clientes.assign(segmento=clusters)
profile=perfiles.groupby('segmento').agg(
    clientes=('cliente_id','size'),
    recencia_mediana=('recencia_dias','median'),
    frecuencia_mediana=('compras_anuales','median'),
    gasto_mediano_ARS=('gasto_anual_ARS','median'),
    descuento_mediano=('descuento_fraccion','median'),
    digital_mediano=('uso_digital','median')).sort_index()
profile['participacion']=profile['clientes']/len(clientes)
audit_table(profile)
# Lectura orientativa: las acciones son HIPÓTESIS, no decisiones automatizadas.
acciones=[]
for grupo,row in profile.iterrows():
    if row['recencia_mediana']>profile['recencia_mediana'].median()*1.35:
        nombre='Compradores alejados';idea='Ensayo controlado de reactivación; medir uplift y margen'
    elif row['gasto_mediano_ARS']>profile['gasto_mediano_ARS'].median()*1.35:
        nombre='Alto valor observado';idea='Probar beneficios relacionales y evitar descuentos innecesarios'
    elif row['descuento_mediano']>profile['descuento_mediano'].median()*1.15:
        nombre='Sensibles al descuento';idea='Comparar promoción vs contenido sin promoción con control'
    else:
        nombre='Compra recurrente';idea='Probar recomendaciones de producto; monitorear frecuencia'
    acciones.append({'segmento':int(grupo),'etiqueta_operativa_provisional':nombre,
                     'hipotesis_de_accion':idea})
audit_table(pd.DataFrame(acciones))
stdprofiles=transformed.assign(segmento=clusters).groupby('segmento').mean()
stdprofiles=pd.DataFrame(rfm_scaler.transform(stdprofiles[features]),
                       index=stdprofiles.index,columns=features)
fig,axes=plt.subplots(1,2,figsize=(13,4.6))
axes[0].bar(profile.index.astype(str),profile['clientes'],color=COLORS[:K_BUSINESS]);axes[0].set(title='Tamaño de los segmentos',xlabel='Identificador',ylabel='Clientes')
im=axes[1].imshow(stdprofiles,cmap='RdBu_r',vmin=-1.5,vmax=1.5)
axes[1].set(title='Perfil relativo estandarizado',xticks=range(len(features)),yticks=range(K_BUSINESS))
axes[1].set_xticklabels([x.replace('_','\n') for x in features],rotation=22,ha='right',fontsize=8)
axes[1].set_yticklabels([f'Grupo {i}' for i in profile.index]);fig.colorbar(im,ax=axes[1],shrink=.8)
fig.tight_layout();plt.show()

### Experimento 24 · Proyección para visualizar y contrastar con Ward

La proyección a 2D solo muestra parte de la varianza. Que dos puntos parezcan cerca en el gráfico de PCA **no implica** que tengan poca distancia en seis dimensiones. Indicamos explícitamente cuánta varianza explica la proyección.

In [ ]:
rfm_pca=PCA(n_components=2).fit(Xrfm)
proj=rfm_pca.transform(Xrfm)
fig,axes=plt.subplots(1,2,figsize=(13,4.6))
scatter_groups(axes[0],proj,clusters,'K-Means sobre seis variables',s=13)
scatter_groups(axes[1],proj,labels_ward,'Ward sobre seis variables',s=13)
for ax in axes:
    ax.set_xlabel(f'PC1 ({rfm_pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({rfm_pca.explained_variance_ratio_[1]:.1%})')
fig.tight_layout();plt.show()
print('Varianza proyectada en 2D:',f'{rfm_pca.explained_variance_ratio_.sum():.1%}')

### Experimento 25 · Asignar clientes nuevos sin reentrenar (producción)

K-Means dispone de una regla de asignación por cercanía al centroide; el clustering jerárquico aglomerativo **no implementa `predict` directamente** en scikit-learn. Para una política productiva hay que documentar y validar una regla aparte (medoides, vecinos, centroides de cada grupo, etc.). **No debemos recalcular `fit_transform` para cada cliente nuevo**: sería una fuga de esquema entre lotes.

In [ ]:
clientes_nuevos=pd.DataFrame({
 'cliente_id':['N001','N002','N003'],
 'recencia_dias':[6,122,37],
 'compras_anuales':[14,1,5],
 'gasto_anual_ARS':[310000,7200,69000],
 'descuento_fraccion':[.09,.41,.25],
 'antiguedad_meses':[38,3,15],
 'uso_digital':[.91,.18,.64]})
new_scaled=rfm_scaler.transform(customer_transform(clientes_nuevos))
new_labels=seg_model.predict(new_scaled)
distance_centers=np.min(cdist(new_scaled,seg_model.cluster_centers_),axis=1)
resultados_nuevos=clientes_nuevos.assign(segmento_predicho=new_labels,
                                         distancia_centroide=distance_centers)
audit_table(resultados_nuevos)
assert len(new_labels)==len(clientes_nuevos)

### Experimento 26 · Un caso de negocio con hipótesis y control

Supongamos que ofrecer una promoción cuesta ARS 1.200 por contacto y que una conversión incremental aporta ARS 7.000 de margen. El equilibrio está en $p_{uplift} > 1200/7000$. **Estas cifras son supuestos ilustrativos**, no tasas estimadas con el clustering. La conversión observada en un segmento no identifica el efecto de una intervención: hace falta asignación aleatoria o un diseño causal defendible.

In [ ]:
COSTO=1200; MARGEN=7000
hipotesis_uplift=pd.Series(np.linspace(.08,.34,K_BUSINESS),index=range(K_BUSINESS),name='uplift_hipotetico')
val=profile[['clientes','participacion']].copy()
val['uplift_sup_zero_control']=hipotesis_uplift
val['valor_neto_contacto_ARS']=val['uplift_sup_zero_control']*MARGEN-COSTO
val['valor_si_contactar_todos_ARS']=val['valor_neto_contacto_ARS']*val['clientes']
audit_table(val)
print('Tasa de uplift necesaria para cubrir costos:',f'{COSTO/MARGEN:.1%}')
print('Interpretación: es sensibilidad financiera a SUPUESTOS, no evaluación causal de una campaña.')

### Experimento 27 · Monitoreo: población nueva, deriva y límites

El número de clientes en cada grupo puede cambiar por estacionalidad, mercado o reglas de medición. Comparar distribuciones de asignación y distancias es un primer control, no prueba de degradación. Incluimos una medida simple **PSI** sobre proporciones de grupos, con suavizado para evitar logaritmos de cero, y una referencia de asignación compatible con datos nuevos.

In [ ]:
future,_=make_customers(n=420,seed=104)
future['recencia_dias']=np.maximum(0,(future['recencia_dias']*1.25).round()).astype(int)
future['gasto_anual_ARS']=(future['gasto_anual_ARS']*1.10).round()
future_z=rfm_scaler.transform(customer_transform(future))
future_labels=seg_model.predict(future_z)
p_train=np.bincount(clusters,minlength=K_BUSINESS)/len(clusters)
p_new=np.bincount(future_labels,minlength=K_BUSINESS)/len(future_labels)
eps=.0001
psi=float(np.sum((p_new-p_train)*np.log((p_new+eps)/(p_train+eps))))
drift=pd.DataFrame({'grupo':range(K_BUSINESS),'porcentaje_entrenamiento':p_train*100,
                    'porcentaje_nuevos':p_new*100,'dif_puntos':(p_new-p_train)*100})
audit_table(drift)
print(f'PSI de mezcla de segmentos: {psi:.3f} (indicador descriptivo; umbrales requieren calibración contextual)')
assert np.isfinite(psi)

---
## 07 · Panel interactivo opcional de experimentación

Este panel **no es obligatorio** para ejecutar las celdas. Si `ipywidgets` está instalado, aparece una interfaz con controles de K, escalado y algoritmo. En caso contrario se ejecuta una comparación estática útil. En Colab se puede instalar opcionalmente `ipywidgets` si no aparece; no hay instalaciones automáticas que interrumpan el laboratorio.

In [ ]:
def panel_experimento(k=4,metodo='K-Means',escala='Standard'):
    X0=customer_transform(clientes)
    op={'Standard':StandardScaler(),'Robust':RobustScaler(),'MinMax':MinMaxScaler()}[escala]
    X=op.fit_transform(X0)
    if metodo=='K-Means':lab=KMeans(n_clusters=int(k),n_init=12,random_state=SEED).fit_predict(X)
    else:lab=AgglomerativeClustering(n_clusters=int(k),linkage='ward').fit_predict(X)
    xy=PCA(n_components=2).fit_transform(X)
    fig,ax=plt.subplots(figsize=(7,4))
    scatter_groups(ax,xy,lab,f'{metodo} · K={k} · {escala}',s=14)
    plt.show()
    print(f'Silhouette = {silhouette_score(X,lab):.3f} · grupos = {len(set(lab))}')

try:
    import ipywidgets as widgets
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    controls={'k':widgets.IntSlider(value=4,min=2,max=7,step=1,description='K'),
       'metodo':widgets.Dropdown(options=['K-Means','Ward'],description='Método'),
       'escala':widgets.Dropdown(options=['Standard','Robust','MinMax'],description='Escala')}
    display(widgets.interactive(panel_experimento,**controls))
except ImportError:
    print('ipywidgets es opcional y no está instalado: ejemplo fijo reproducible.')
    panel_experimento(k=4,metodo='K-Means',escala='Standard')

---
## 08 · Pruebas automatizadas y contrato de reproducibilidad

No basta con generar gráficos: comprobamos invariantes matemáticas para detectar regresiones en futuras actualizaciones del notebook. Las verificaciones **no sustituyen** una revisión estadística de pertinencia o validación de producción.

In [ ]:
assert Xrfm.shape==(len(clientes),len(features))
assert np.isfinite(Xrfm).all()
assert np.allclose(Xrfm.mean(axis=0),0,atol=1e-10)
assert np.allclose(Xrfm.var(axis=0),1,atol=1e-10)
assert np.isclose(pca9.explained_variance_ratio_.sum(),1)
assert np.all(np.diff(mses)<=1e-8)
assert all(np.isfinite([x['silhouette'],x['inercia']]).all() for x in metrics)
assert all(np.unique(fcluster(z,3,criterion='maxclust')).size<=3 for z in Zs.values())
assert len(np.unique(seg_model.labels_))==K_BUSINESS
assert np.allclose(seg_model.transform(Xrfm),cdist(Xrfm,seg_model.cluster_centers_))
assert len(resultados_nuevos)==3
checks=pd.DataFrame([
 ('PCA: autovalores SVD y covarianza',np.allclose(lam_svd,lam_eig)),
 ('PCA: varianza total ≈ 100%',np.isclose(pca9.explained_variance_ratio_.sum(),1)),
 ('PCA: reconstrucción no empeora con q',np.all(np.diff(mses)<=1e-8)),
 ('K-Means: inercia no creciente por iteración',np.all(np.diff([h['inercia_despues'] for h in hist])<=1e-8)),
 ('K-Means: K evaluados 2 a 8',len(kmmetrics)==7),
 ('Jerárquico: 4 métodos / dendrogramas',len(Zs)==4),
 ('RFM: no hay nulos ni infinitos',np.isfinite(Xrfm).all()),
 ('RFM: K grupos válidos',len(np.unique(clusters))==K_BUSINESS),
 ('Producción: nuevos clientes sin reentrenar',len(new_labels)==len(clientes_nuevos)),
 ('Monitoreo: PSI finito',np.isfinite(psi)),
],columns=['Prueba','Aprobada'])
audit_table(checks)
assert checks['Aprobada'].all()
print('10/10 comprobaciones numéricas: aprobadas')

## 09 · Exportación de resultados, modelo y bitácora

El informe debe ser **auditable**: parámetros, variables, escala, método y fecha. Guardamos un CSV de segmentos, otro de perfiles, un JSON de decisiones y el objeto entrenado con su escalador. En Google Colab los archivos aparecen en el navegador lateral de archivos; no se exportan registros personales porque los datos son sintéticos.

In [ ]:
import joblib
from datetime import datetime,timezone
OUT=Path('m10_resultados');OUT.mkdir(exist_ok=True)
clientes.assign(segmento=clusters).to_csv(OUT/'clientes_segmentados.csv',index=False,encoding='utf-8-sig')
profile.to_csv(OUT/'perfiles_gerenciales.csv',encoding='utf-8-sig')
rfm_metrics.to_csv(OUT/'diagnosticos_K.csv',index=False,encoding='utf-8-sig')
joblib.dump({'scaler':rfm_scaler,'modelo':seg_model,'features':features,
             'transformacion':'log1p recencia, frecuencia, gasto y antiguedad'},
             OUT/'pipeline_segmentacion.joblib')
registro={
 'modulo':'10','curso':'Ciencia de Datos para Decisiones de Negocio',
 'autor':'Sergio Gevatschnaider','semilla':SEED,'n_clientes':len(clientes),
 'modelo':'KMeans','k':K_BUSINESS,'n_init':35,
 'variables':features,'silhouette':float(silhouette_score(Xrfm,clusters)),
 'contraste':'Ward sobre espacio escalado',
 'diferencia_algoritmos_ARI':float(adjusted_rand_score(clusters,labels_ward)),
 'advertencia':'Datos sintéticos. Las campañas requieren evaluación causal, privacidad y control.',
 'versiones':versions,'actualizado_utc':datetime.now(timezone.utc).isoformat()}
(OUT/'registro_decision.json').write_text(json.dumps(registro,indent=2,ensure_ascii=False),encoding='utf-8')
print('ARCHIVOS CREADOS:')
for p in sorted(OUT.iterdir()):print('-',p.name,':',p.stat().st_size,'bytes')
assert all((OUT/f).is_file() for f in ['clientes_segmentados.csv','perfiles_gerenciales.csv',
                                      'diagnosticos_K.csv','pipeline_segmentacion.joblib','registro_decision.json'])

## 10 · Preguntas de desarrollo y evaluación (respuestas razonadas)

Estas preguntas sirven como preparación de parcial, defensa de proyecto y discusión de resultados. Intentá responder **antes** de abrir los desarrollos. No se requieren cálculos manuales extensos.

<details><summary><b>1. ¿Por qué PCA no determina cuántos clusters existen?</b></summary><p>PCA busca direcciones de máxima varianza total de una nube, sin conocer etiquetas ni densidad. Dos grupos compactos pueden separarse en una dirección de poca varianza y desaparecer en una proyección agresiva. Además, las mayores varianzas pueden reflejar escalas o valores extremos. Para defender clusters se requieren métricas de separación, estabilidad, perfiles interpretables y utilidad.</p></details>
<details><summary><b>2. ¿Qué distingue scores, componentes, eigenvalores y loadings?</b></summary><p>Los componentes son direcciones unitarias en el espacio de variables; los scores son coordenadas de cada observación proyectada; los autovalores cuantifican dispersión a lo largo de cada dirección; los loadings se usan para explicar relaciones de las variables con los componentes (su definición exacta depende de la convención). El signo del componente es arbitrario.</p></details>
<details><summary><b>3. ¿Cuándo conviene PCA antes de K-Means?</b></summary><p>Cuando existen variables muy redundantes, ruido de alta dimensión o costos de cálculo elevados, la reducción puede simplificar la métrica. No garantiza mejor segmentación: componentes descartados pueden codificar grupos minoritarios relevantes. Hay que comparar K-Means con y sin PCA, medir estabilidad y documentar el criterio de selección de componentes sin usar información de validación indebidamente.</p></details>
<details><summary><b>4. ¿Por qué la inercia baja si K aumenta?</b></summary><p>Más centroides permiten representar cada punto al menos tan cerca como antes, puesto que el conjunto de particiones posibles contiene configuraciones más flexibles. Por ello minimizar inercia sin penalizar complejidad terminaría con un centro por observación. El codo es heurístico y debe contrastarse con silhouette y criterios operacionales.</p></details>
<details><summary><b>5. ¿Qué diferencia hay entre random y K-Means++?</b></summary><p>La inicialización aleatoria escoge centros sin optimizar su cobertura espacial; K-Means++ selecciona el primer centro y favorece posteriores centros alejados de los ya elegidos mediante probabilidades basadas en distancia cuadrática. Suele reducir configuraciones malas, pero ambas pueden terminar en óptimos locales. Se necesitan reinicios y sensibilidad a semillas.</p></details>
<details><summary><b>6. ¿Cómo se interpretan valores negativos de silhouette?</b></summary><p>Significan que la disimilitud media del punto hacia otro grupo es menor que su disimilitud media dentro del grupo actual, con la métrica utilizada. Pueden indicar una asignación conflictiva, solapamiento o estructura inapropiada. No se puede inferir automáticamente que ese punto sea un error de captura.</p></details>
<details><summary><b>7. ¿Qué es chaining y qué linkage lo favorece?</b></summary><p>Es el fenómeno por el que se forman cadenas de elementos conectados por vecinos cercanos, incluso si los extremos están distantes. Single linkage es sensible a este efecto porque usa la menor distancia entre pares de dos grupos. Complete contrasta con la distancia máxima y tiende a ser más compacto; average promedia; Ward minimiza un incremento de dispersión euclídea.</p></details>
<details><summary><b>8. ¿La altura de Ward es una distancia entre dos puntos?</b></summary><p>No debe interpretarse así. Ward combina clusters para minimizar el aumento de la suma de cuadrados interna, y su altura depende de la convención de linkage adoptada. Para interpretar una fusión hay que declarar método, unidades, escalado y criterio; comparar alturas numéricas entre métodos sin contexto conduce a errores.</p></details>
<details><summary><b>9. ¿Puede DBSCAN ser superior a K-Means?</b></summary><p>Puede recuperar grupos curvos y señalar ruido sin elegir K de antemano, como las dos lunas. Sin embargo, exige seleccionar eps y min_samples; presenta problemas con densidades variables y distancias poco informativas en alta dimensión. No existe un algoritmo universalmente mejor: depende de geometría, negocio y validación.</p></details>
<details><summary><b>10. ¿Cómo comparo particiones si cambian los números de etiquetas?</b></summary><p>Los identificadores de clusters son arbitrarios. El Adjusted Rand Index compara relaciones de pares —juntos/separados— corregidas por azar bajo sus supuestos y es invariante ante permutación de nombres. ARI igual a uno representa la misma partición; valores alrededor de cero son concordancia comparable al azar. Para estabilidad se comparan asignaciones de observaciones en común.</p></details>
<details><summary><b>11. ¿Cómo asigno clientes nuevos si utilicé clustering jerárquico?</b></summary><p>El aglomerativo de sklearn no ofrece una regla de predict lista para nuevos datos. Se puede diseñar una aproximación entrenable —por ejemplo, asignación al centroide o medoide de cada grupo— y validarla, sin confundirla con una extensión exacta del dendrograma. K-Means sí permite usar sus centros; en todos los casos deben reutilizarse transformaciones entrenadas.</p></details>
<details><summary><b>12. ¿Por qué un buen silhouette no demuestra rentabilidad?</b></summary><p>Silhouette mide proximidad geométrica relativa; una campaña afecta conductas económicas, costos y restricciones. Incluso un segmento muy compacto puede responder igual que los demás. La mejora incremental se estima mediante experimentos o diseños causales; se compara margen incremental esperado con el costo de tratar el segmento y riesgos de privacidad o discriminación.</p></details>
<details><summary><b>13. ¿Qué significa que PCA cambie ante un outlier?</b></summary><p>La varianza es sensible a distancias cuadráticas, por lo que un extremo puede dominar orientaciones. Debe verificarse si el extremo es legítimo, revisar captura, hacer análisis de influencia y comparar técnicas robustas o transformaciones justificadas. RobustScaler no vuelve robusta por sí misma la optimización de PCA.</p></details>
<details><summary><b>14. ¿Qué es fuga de información en la etapa de escalado?</b></summary><p>Ocurre cuando se calcula una transformación con datos que deberían representar validación, futuro o producción. Incluso sin variable objetivo, la evaluación fuera de muestra puede resultar demasiado favorable si la escala o imputación incorporaron información posterior. La solución es ajustar el preprocesamiento en entrenamiento y aplicar transform a datos nuevos mediante el mismo pipeline.</p></details>
<details><summary><b>15. ¿Qué se debería monitorear luego de lanzar una segmentación?</b></summary><p>Distribución y tamaño de segmentos, distancias a centroides, tasas de ruido, cambios de datos y faltantes, evolución temporal, estabilidad de perfiles, acciones y respuesta incremental evaluada con control. Se establecen disparadores de reentrenamiento y responsables. Un PSI alto es señal de cambio, no diagnóstico causal automático.</p></details>

## 11 · Desafío final y rúbrica de evaluación

**Consigna integradora.** Una empresa de comercio electrónico quiere lanzar tres propuestas diferenciales con presupuesto limitado. Debés defender un plan y documentar las decisiones antes de usar la segmentación en producción.

| Dimensión evaluada | Excelente | Insuficiente |
|---|---|---|
| Representación y distancia | Variables, escala, unidades y datos faltantes justificados | Se aplica K-Means directamente a montos crudos |
| PCA y visualización | Varianza, cargas, reconstrucción y pérdida explicadas | Se confunde PC1/PC2 con grupos verdaderos |
| Elección del método | K e inicializaciones, linkages y DBSCAN contrastados | Se afirma que un algoritmo siempre gana |
| Calidad y estabilidad | Métricas internas, perturbaciones y remuestreo documentados | Se informa solo inercia o un gráfico atractivo |
| Decisión empresarial | Perfil, tamaño, alcanzabilidad, costo y prueba con control | Se equipara agrupamiento con uplift causal |
| Reproducibilidad | Semillas, versiones, exportaciones y seguimiento | Resultados no regenerables |

**Registro de decisión:** informá 1) contexto, 2) variables, 3) alternativas, 4) diagnóstico de validación, 5) riesgos, 6) acción propuesta, 7) responsable, 8) experimento de impacto, 9) plan de monitoreo y 10) condiciones para revertir la decisión.

**Criterio de éxito final:** otro estudiante debe poder ejecutar **todo el notebook** y explicar en lenguaje estadístico y económico qué aprendió, qué no demostró y qué acción puede poner a prueba.

<div style="background:#0b263b;border:1px solid #36c7b0;border-radius:15px;padding:22px 24px;color:#eaf8ff">
<h2 style="margin-top:0;color:#7de4d6">Cierre · De patrones a decisiones</h2>
<p>Una representación geométrica no es automáticamente un segmento comercial. PCA proyecta, K-Means optimiza centroides, el clustering jerárquico revela fusiones a diferentes escalas y DBSCAN modela densidad. Las métricas ofrecen evidencia parcial. Solo una evaluación de estabilidad, interpretación, costos y causalidad puede convertir esa estructura en decisiones defendibles.</p>
<p><b>Próxima lectura:</b> <a style="color:#87c9ff" href="https://sgevatschnaider.github.io/data-science-business-decisions/modulos/10-clustering/glosario.html">glosario razonado de 132 conceptos</a> · <a style="color:#87c9ff" href="https://sgevatschnaider.github.io/data-science-business-decisions/modulos/10-clustering/cuestionario.html">cuestionario de 128 preguntas</a>.</p>
<p style="font-size:12px;color:#bddde7">Material elaborado por el profesor Sergio Gevatschnaider · Versión académica integral · Módulo 10</p>
</div>